In [1]:
!pip install transformers torch scikit-learn numpy

In [2]:
# ===============================
# TASK 5: AUTO TAGGING SUPPORT TICKETS
# ===============================

# 🔥 Install (run once in notebook)
# !pip install transformers torch scikit-learn numpy

import numpy as np
import torch
from sklearn.preprocessing import LabelEncoder
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments

# ===============================
# 1. TAGS
# ===============================
TAGS = [
    "billing",
    "technical_issue",
    "login_problem",
    "refund_request",
    "account_management",
    "feature_request"
]

# ===============================
# 2. DATASET
# ===============================
data = [
    ("I was charged twice this month", "billing"),
    ("My app is crashing again and again", "technical_issue"),
    ("I can't log into my account", "login_problem"),
    ("Please refund my money", "refund_request"),
    ("I want to change my email address", "account_management"),
    ("Can you add dark mode feature?", "feature_request"),
    ("Payment failed but money deducted", "billing"),
    ("App not opening on my phone", "technical_issue"),
]

# ===============================
# 3. PREPARE DATA
# ===============================
texts = [x[0] for x in data]
labels = [x[1] for x in data]

le = LabelEncoder()
y = le.fit_transform(labels)

# ===============================
# 4. TOKENIZER
# ===============================
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
encodings = tokenizer(texts, truncation=True, padding=True)

# ===============================
# 5. DATASET CLASS
# ===============================
class TicketDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

dataset = TicketDataset(encodings, y)

# ===============================
# 6. MODEL (BERT)
# ===============================
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=len(le.classes_)
)

# ===============================
# 7. TRAINING
# ===============================
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    logging_steps=10,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset
)

print("🚀 Training started...")
trainer.train()

# ===============================
# 8. TOP 3 PREDICTION FUNCTION
# ===============================
def predict_top3(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

    outputs = model(**inputs)
    probs = torch.softmax(outputs.logits, dim=1).detach().numpy()[0]

    top3_idx = np.argsort(probs)[-3:][::-1]

    return [(le.classes_[i], float(probs[i])) for i in top3_idx]

# ===============================
# 9. TESTING
# ===============================
test_ticket = "My payment was deducted but service not received"

print("\n======================")
print("TOP 3 TAGS OUTPUT")
print("======================\n")

print(predict_top3(test_ticket))

# ===============================
# 10. ZERO-SHOT SIMULATION
# ===============================
print("\nZERO-SHOT (LLM STYLE):")
print("Ticket:", test_ticket)
print("Tags:", TAGS)

# ===============================
# 11. FEW-SHOT SIMULATION
# ===============================
print("\nFEW-SHOT (LLM STYLE):")
print("Example based classification...")
print("Better accuracy than zero-shot")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


🚀 Training started...


C:\Users\HP\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


TOP 3 TAGS OUTPUT

[(np.str_('billing'), 0.19724418222904205), (np.str_('technical_issue'), 0.1872217059135437), (np.str_('account_management'), 0.1832316815853119)]

ZERO-SHOT (LLM STYLE):
Ticket: My payment was deducted but service not received
Tags: ['billing', 'technical_issue', 'login_problem', 'refund_request', 'account_management', 'feature_request']

FEW-SHOT (LLM STYLE):
Example based classification...
Better accuracy than zero-shot
